## What is happening in the next code?
This code finds the project's main folder (one level above the current folder) and builds the path to `data/manifests/unique_images.csv`.
It loads that CSV file into a table called `clean_df`, which is the cleaned list of unique images saved earlier.
Then it prints how many images were loaded and how many duplicate hashes exist (it should be 0).

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
manifest_path = PROJECT_ROOT / "data" / "manifests" / "unique_images.csv"
clean_df = pd.read_csv(manifest_path)

print("Unique images loaded:", len(clean_df))
print("Duplicate hashes:", clean_df["sha256"].duplicated().sum())

Unique images loaded: 3512
Duplicate hashes: 0


## What is happening in the next code?
This code splits the clean images into three parts: 70% for training, 15% for validation, and 15% for testing.
It uses `stratify` so every part keeps the same class balance, and `random_state=42` so the split is the same every time you run it.
Then it prints the size of each part and checks that no file fingerprint appears in more than one part, so there is no duplicate leakage between them.

In [2]:
from sklearn.model_selection import train_test_split

train_df, remaining_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    stratify=remaining_df["label"],
    random_state=42,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

assert set(train_df["sha256"]).isdisjoint(val_df["sha256"])
assert set(train_df["sha256"]).isdisjoint(test_df["sha256"])
assert set(val_df["sha256"]).isdisjoint(test_df["sha256"])
print("Exact duplicates shared between splits: 0")

Train: 2458
Validation: 527
Test: 527
Total: 3512
Exact duplicates shared between splits: 0


## What is happening in the next code?
This code counts how many images of each class are in the train, validation, and test parts, and puts the counts in one table.
Missing counts are filled with 0, and the classes are sorted by name.
It also adds a `total` column and shows the table, so you can check that every class is split in a balanced way.

In [3]:
class_counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int).sort_index()

class_counts["total"] = class_counts.sum(axis=1)
display(class_counts)

,train,validation,test,total
label,,,,
ambulance,285,61,62,408
autobus,364,78,78,520
kamyun,306,66,65,437
kamyunet,430,92,92,614
minibus,304,65,65,434
savari,246,53,53,352
taxi,244,52,52,348
vanet,279,60,60,399


## What is happening in the next code?
This code puts the three parts (train, validation, test) in a dictionary and loops over them one by one.
Each part is sorted by label and path, then saved as its own CSV file (`train.csv`, `validation.csv`, `test.csv`) in the `data/manifests` folder.
Then it reads each saved file back and prints the number of rows, so you can check that the save worked.

In [4]:
split_files = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

for name, split_df in split_files.items():
    output_path = PROJECT_ROOT / "data" / "manifests" / f"{name}.csv"
    split_df.sort_values(["label", "path"]).to_csv(output_path, index=False)
    print(f"{name}: {len(pd.read_csv(output_path))} rows saved")

train: 2458 rows saved
validation: 527 rows saved
test: 527 rows saved


## What is happening in the next code?
This code loads the three saved CSV files (train, validation, test) from the `data/manifests` folder and joins them into one big table.
It checks the saved data: how many rows there are in total, how many unique fingerprints (`sha256`) exist, and how many image paths point to files that are missing on disk.
Then it prints the list of classes inside each split, so you can confirm every split has all the classes.

In [5]:
manifest_dir = PROJECT_ROOT / "data" / "manifests"

saved_splits = {
    name: pd.read_csv(manifest_dir / f"{name}.csv")
    for name in ("train", "validation", "test")
}

all_saved = pd.concat(saved_splits.values(), ignore_index=True)

print("Total saved rows:", len(all_saved))
print("Unique hashes:", all_saved["sha256"].nunique())
print("Missing image paths:", sum(not Path(p).is_file() for p in all_saved["path"]))
print("Classes in each split:")
for name, split_df in saved_splits.items():
    print(name, sorted(split_df["label"].unique()))

Total saved rows: 3512
Unique hashes: 3512
Missing image paths: 0
Classes in each split:
train ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
validation ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
test ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
